# Topic coverage in bio.tools

Counts how many tools have any EDAM topic, and how many are tagged with Omics, Metabolomics and Proteomics.

Input: `data/biotools_topics_operations.csv`, made by `extract_topics_operations.py`.

In [1]:
import duckdb

CSV = "data/biotools_topics_operations.csv"

OMICS = "topic_3391"         # EDAM "Omics"
METABOLOMICS = "topic_3172"  # EDAM "Metabolomics"
PROTEOMICS = "topic_0121"    # EDAM "Proteomics"

# Turn the "|"-separated topic ids into a list per tool.
duckdb.sql(f"""
    CREATE OR REPLACE VIEW tools AS
    SELECT biotoolsID, name, topics, "n topics"::int AS n_topics,
           string_split(coalesce("topic ids", ''), '|') AS topic_ids
    FROM read_csv('{CSV}', all_varchar = true)
""")

## Counts

`has omics` means the EDAM topic *Omics* itself. `topic name contains "omics"` is broader and also catches Genomics, Transcriptomics and so on.

In [2]:
duckdb.sql(f"""
    UNPIVOT (
    SELECT
        count(*)                                                    AS "all tools",
        count(*) FILTER (n_topics > 0)                              AS "has any topic",
        count(*) FILTER (list_contains(topic_ids, '{OMICS}'))       AS "has omics",
        count(*) FILTER (topics ILIKE '%omics%')                    AS "topic name contains 'omics'",
        count(*) FILTER (list_contains(topic_ids, '{METABOLOMICS}')) AS "has metabolomics",
        count(*) FILTER (list_contains(topic_ids, '{PROTEOMICS}'))  AS "has proteomics",
        count(*) FILTER (list_contains(topic_ids, '{METABOLOMICS}')
                     AND list_contains(topic_ids, '{PROTEOMICS}'))  AS "has metabolomics and proteomics"
    FROM tools
    ) ON COLUMNS(*) INTO NAME "check" VALUE "tools"
""").show()

┌─────────────────────────────────┬───────┐
│              check              │ tools │
│             varchar             │ int64 │
├─────────────────────────────────┼───────┤
│ all tools                       │ 34488 │
│ has any topic                   │ 32704 │
│ has omics                       │   128 │
│ topic name contains 'omics'     │  8236 │
│ has metabolomics                │   812 │
│ has proteomics                  │  2050 │
│ has metabolomics and proteomics │   150 │
└─────────────────────────────────┴───────┘



## Tools tagged with both Metabolomics and Proteomics

In [3]:
duckdb.sql(f"""
    SELECT biotoolsID, name, topics
    FROM tools
    WHERE list_contains(topic_ids, '{METABOLOMICS}')
      AND list_contains(topic_ids, '{PROTEOMICS}')
    ORDER BY biotoolsID
""").show(max_rows=200)

┌──────────────────────────────────────────┬─────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                biotoolsID                │                      name                       │                                                                                                            topics                                                                                                            │
│                 varchar                  │                     varchar                     │                                                                                                           varchar                                                                                                            │
├──────────────────────────────────────────┼──